# 06. Shortcut analysis

LightGBM scores very high on PaySim. This notebook checks where the score comes from:
a plain rule, ablations of the features that reproduce it, and the recipient-history features.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_score, recall_score

from src.features import (
    FINAL_CONTINUOUS, FINAL_BINARY, CATEGORICAL, FINAL_FEATURES, add_recipient_history
)
from src.split import make_masks
from src.models import make_lgbm, prepare_lgbm_frame

In [ ]:
model_df = pd.read_parquet("../data/processed/model_df.parquet")
model_df = add_recipient_history(model_df)   # features about earlier operations on the recipient account

train_mask, val_mask, test_mask = make_masks(model_df)

y_train = model_df.loc[train_mask, "is_fraud"]
y_val = model_df.loc[val_mask, "is_fraud"]
y_test = model_df.loc[test_mask, "is_fraud"]

## 1. A plain rule: the whole balance is transferred

`full_balance_attempt == 1` means that the amount equals the sender's balance.

In [ ]:
for name, mask in [("train", train_mask), ("validation", val_mask), ("test", test_mask)]:
    d = model_df.loc[mask]
    fraud = d[d["is_fraud"] == 1]
    honest = d[d["is_fraud"] == 0]
    print(
        f"{name:11s} full_balance_attempt: fraud {fraud['full_balance_attempt'].mean() * 100:.1f}%, "
        f"legitimate {honest['full_balance_attempt'].mean() * 100:.1f}%"
    )

d = model_df.loc[test_mask]
rule = d["full_balance_attempt"]
print(
    "Rule on test: precision", round(precision_score(d["is_fraud"], rule), 3),
    "recall", round(recall_score(d["is_fraud"], rule), 3),
)

## 2. What the rule misses

The frauds that the rule does not catch seem to have an amount of exactly 10,000,000, which looks like a cap
on a single transaction in the simulator (the balance is larger than the cap, so the whole balance cannot be moved).

In [ ]:
d = model_df.loc[test_mask]

missed = (d["is_fraud"] == 1) & (d["full_balance_attempt"] == 0)
print("Frauds missed by the rule:", int(missed.sum()), "of", int(d["is_fraud"].sum()))

d.loc[missed, ["transaction_type", "amount", "old_balance_orig", "old_balance_dest", "dest_prev_ops"]].describe().round(1)

In [ ]:
cap = d["amount"] == 10_000_000

print("Operations with an amount of exactly 10 million:", int(cap.sum()))
print("Fraud share among them:", round(d.loc[cap, "is_fraud"].mean(), 3))
print("Frauds with that amount:", int((cap & (d["is_fraud"] == 1)).sum()), "of", int(d["is_fraud"].sum()))

rule_v2 = ((d["full_balance_attempt"] == 1) | cap).astype(int)
print(
    "Rule v2 (whole balance OR amount = 10M): precision",
    round(precision_score(d["is_fraud"], rule_v2), 3),
    "recall", round(recall_score(d["is_fraud"], rule_v2), 3),
)

## 3. Ablations: which features does LightGBM rely on?

Removing the ratio and `full_balance_attempt` is not enough: `log_amount` and `log_old_balance_orig` together
reproduce "amount equals balance". Each variant below is trained with the same settings.

In [ ]:
def fit_and_score(continuous, binary=FINAL_BINARY):
    cols = continuous + binary + CATEGORICAL
    model = make_lgbm()
    model.fit(prepare_lgbm_frame(model_df.loc[train_mask], cols), y_train)

    scores = {}
    for part, mask, y in [("val", val_mask, y_val), ("test", test_mask, y_test)]:
        p = model.predict_proba(prepare_lgbm_frame(model_df.loc[mask], cols))[:, 1]
        scores[f"PR-AUC {part}"] = round(average_precision_score(y, p), 3)
    return model, scores


RATIO = "log_amount_to_balance_ratio"

variants = {
    "final features": FINAL_CONTINUOUS,
    "final + ratio": FINAL_CONTINUOUS + [RATIO],
    "without log_old_balance_orig": [c for c in FINAL_CONTINUOUS if c != "log_old_balance_orig"],
    "without log_amount": [c for c in FINAL_CONTINUOUS if c != "log_amount"],
    "without both": [c for c in FINAL_CONTINUOUS if c not in ("log_old_balance_orig", "log_amount")],
}

ablation_rows, models = [], {}
for name, continuous in variants.items():
    model, scores = fit_and_score(continuous)
    models[name] = model
    ablation_rows.append({"experiment": name, **scores})

pd.DataFrame(ablation_rows)

In [ ]:
model = models["final features"]
importance = pd.Series(
    model.booster_.feature_importance(importance_type="gain"),
    index=model.booster_.feature_name(),
).sort_values(ascending=False)

(importance / importance.sum()).round(3)

## 4. Recipient history

Does the recipient account already have a history? Only strictly earlier steps are counted (no future).

In [ ]:
print(model_df.groupby("is_fraud")["dest_prev_ops"].describe().round(2))
print()
print("Share of operations to a recipient with no earlier history:")
print(model_df.groupby("is_fraud")["dest_is_new"].mean().round(3))

In [ ]:
model_df["period"] = pd.cut(model_df["step"], bins=[0, 200, 400, 600, 743])

model_df.groupby(["period", "is_fraud"], observed=True)["dest_prev_ops"].agg(["count", "mean", "median"])

In [ ]:
_, with_history = fit_and_score(
    FINAL_CONTINUOUS + ["log_dest_prev_ops"],
    FINAL_BINARY + ["dest_is_new"],
)
print("Final features + recipient history:", with_history)

## Summary

The score of the strong model comes from a rule of the simulator that the amount and the sender's balance reveal.
Without those two features the score drops a lot. The recipient history differs clearly between fraud and legitimate
operations, but adds nothing on top of the amount and balance.